# CMPU395 Final Project Analysis

Jensen Bergman and Jacob Sellitti - 5/1/26

## Introduction

Constructing an optimal batting order is a complex problem that blends player performance, team context, and strategic decision-making. In this notebook, we take a data-driven approach to this problem using Statcast data from the 2025 MLB season.

Our goal is to model how players are assigned lineup positions based on their offensive performance up to that point in the season. To do this, we implement a two-model framework:

- A player-level model that predicts the probability distribution over lineup positions (1–9) for an individual player
- A team-level model that predicts a full lineup for a group of nine players


In [2]:
# Install required packages
! pip install pandas numpy tqdm pybaseball scikit-learn torch

  Using cached pybaseball-2.2.7-py3-none-any.whl.metadata (11 kB)
  Using cached torch-2.11.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached pynacl-1.6.2-cp38-abi3-macosx_10_10_universal2.whl.metadata (10.0 kB)
  Using cached pyjwt-2.12.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached pybaseball-2.2.7-py3-none-any.whl (426 kB)
Using cached torch-2.11.0-cp313-cp313-macosx_11_0_arm64.whl (80.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.1 MB/s  0:00:00
Using cached fsspec-2026.3.0-py3-none-any.whl (202 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 32.0 MB/s  0:00:01m0:00:0100:01
Using cached pyjwt-2.12.1-py3-none-any.whl (29 kB)
   

In [3]:
# Imports
import pandas as pd
from pybaseball import statcast, playerid_reverse_lookup
import csv
from io import StringIO
import datetime
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import ast
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from scipy.optimize import linear_sum_assignment
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

## Data Collection

Using the pybaseball library, we scraped pitch-by-pitch Statcast data for the 2025 MLB season. This dataset contains detailed information about every pitch, including outcomes (hits, strikeouts, walks), batted ball metrics (launch angle, exit velocity), and game context.

In addition, we incorporate lineup data scraped separately, which provides the batting order for each team in each game. This allows us to map individual player performances to their corresponding lineup positions.

The raw Statcast dataset is highly granular (one row per pitch), so a key challenge is transforming this into meaningful player-level features that evolve over the course of a season.


In [5]:
# LOAD STATCAST DATA FOR 2025 SEASON
season_2025 = statcast('2025-03-01', '2025-11-01')
# Only get regular season events where something happened (not just a ball or strike)
season_2025_events = season_2025[season_2025['events'].notna()]
season_2025_events = season_2025_events[season_2025_events['game_type'] == 'R']
# Filter by batters who had more than 100 total plate appearances
data = season_2025_events[season_2025_events['batter'].isin(
    season_2025_events['batter'].value_counts()[season_2025_events['batter'].value_counts() > 100].index)].copy()

# Convert MLB IDs to Retrosheet IDs for easier lineup merging
unique_players = data['batter'].dropna().unique().tolist()
player_map = playerid_reverse_lookup(unique_players)
player_map = player_map[['key_mlbam', 'key_retro']]
mlb_to_retro = dict(zip(player_map['key_mlbam'], player_map['key_retro']))
data['batter'] = data['batter'].map(mlb_to_retro)
data = data[data['batter'].notna()]

# Sort by date, team, and inning (assending)
season_2025_data = data.sort_values(by=['game_date', 'home_team', 'inning'])
print("Statcast data saved")

# LOAD GAME DATA TO PARSE BATTING ORDERS
with open('../data/gl2025.txt', 'r') as file:
    lines = file.readlines()
parsed = []
for line in lines:
    parsed.extend(list(csv.reader(StringIO(line))))

rows = []
for game in parsed:
    away_lineup = []
    home_lineup = []
    for i in range(9):
        away_lineup.append(game[105 + i*3])
        home_lineup.append(game[132 + i*3])
    date = datetime.datetime.strptime(game[0], '%Y%m%d').strftime('%Y-%m-%d')
    rows.append({
        'date': date,
        'away_lineup': away_lineup,
        'home_lineup': home_lineup
    })

lineups = pd.DataFrame(rows)
print("Lineup data saved")

This is a large query, it may take a moment to complete


/Users/jacob/Library/r-miniconda-arm64/lib/python3.13/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)


Skipping offseason dates


100%|██████████| 232/232 [00:13<00:00, 17.20it/s]


Statcast data saved
Lineup data saved


## Data Preparation and Feature Engineering

To construct meaningful inputs for our models, we aggregate pitch-level data into player-level statistics that reflect performance up to/before each game. This prevents data leakage and better reflects real-world decision-making.

For each player, we compute the following cumulative statistics:
- Batting Average (AVG) - Fraction of hits/at-bats of an individual player. This will be between 0 and 1.
- Slugging Percentage (SLG) - Fraction of total bases/at bats of an individual player, where singles are worth 1, doubles are worth 2, triples are worth 3, and home runs are worth 4. This will be between 0 and 4.
- On base percentage (OBP) - Fraction of times on base (hits + walks + hit by pitches) / plate appearances. Will be between 0 and 1.
- Expected weighted on-base average (xwOBA) - Measures offensive performance based on quality of contact (exit velocity and launch angle), removing the effect of defense and luck.
- Home Runs (HR) - Total number of home runs a batter has hit.
- Runs Batted In (RBI) - Total number of runs a batter has batted in.
- Stolen bases (SB) - Total number of stolen bases a player has.
- Contact percentage - Fraction of total contact (balls put in play) / total number of swings.
- Strikeout percentage (K%) - Fraction of strikeouts/total at-bats.
- Walk percentage (BB%) - Fraction of walks/total at-bats.
- Strikeout/Walk Ratio (K/BB) - Ratio of strikeouts to walks
- Sweet spot percentage - The percentage of the time a hitter hits a ball with a launch angle between 8 and 32 degrees.
- Hard hit percentage (HH%) - The percentage of the time a hitter hits a ball over 95 MPH.

We also implement safeguards such as:
- Safe division to avoid undefined values
- Filtering out early-season data (first 10 games) to allow statistics to stabilize
- Removing pinch hitters (players without consistent lineup positions) to ensure we have a max of 9 players per team per game

Finally, we join this data with lineup information to assign each player their batting order position for each game, and save the completed feature data.


In [7]:
features_df = []

def get_lineup_pos(batter_id, lineups, side):
    for _, row in lineups.iterrows():
        lineup = row[side]
        if isinstance(lineup, str):
            lineup = ast.literal_eval(lineup)
        if batter_id in lineup:
            return lineup.index(batter_id) + 1
    return np.nan

for player_id, group in tqdm(season_2025_data.groupby('batter'), desc="Calculating features"):
    group = group.sort_values('game_date').copy()

    # Flag variables
    group['is_hit'] = group['events'].isin(['single', 'double', 'triple', 'home_run']).astype(int)
    group['is_ab'] = (~group['events'].isin(['walk', 'hit_by_pitch', 'sac_bunt', 'sac_fly'])).astype(int)
    group['is_bb'] = (group['events'] == 'walk').astype(int)
    group['is_k'] = (group['events'] == 'strikeout').astype(int)
    group['is_hr'] = (group['events'] == 'home_run').astype(int)
    group['is_hbp'] = (group['events'] == 'hit_by_pitch').astype(int)
    group['pa'] = group['is_ab'] + group['is_bb'] + group['is_hbp']
    group['is_bip'] = group['events'].isin(['single','double','triple','home_run',
                                            'field_out','force_out','grounded_into_double_play']).astype(int)
    group['tb'] = (
        (group['events'] == 'single').astype(int)
        + 2*(group['events'] == 'double').astype(int)
        + 3*(group['events'] == 'triple').astype(int)
        + 4*(group['events'] == 'home_run').astype(int)
    )

    # Cumulative stats up to that point
    c_hits = group['is_hit'].cumsum().shift(1).fillna(0)
    c_ab = group['is_ab'].cumsum().shift(1).fillna(0)
    c_tb = group['tb'].cumsum().shift(1).fillna(0)
    c_bb = group['is_bb'].cumsum().shift(1).fillna(0)
    c_k = group['is_k'].cumsum().shift(1).fillna(0)
    c_hbp = group['is_hbp'].cumsum().shift(1).fillna(0)
    c_pa = group['pa'].cumsum().shift(1).fillna(0)
    c_hr = group['is_hr'].cumsum().shift(1).fillna(0)
    c_bip = group['is_bip'].cumsum().shift(1).fillna(0)

    # Safe division helper
    def safe_div(num, denom):
        return np.where(denom > 0, num / denom, 0)

    group['AVG'] = safe_div(c_hits, c_ab)
    group['SLG'] = safe_div(c_tb, c_ab)
    group['OBP'] = safe_div(c_hits + c_bb + c_hbp, c_pa)
    group['HR'] = c_hr
    group['K%'] = safe_div(c_k, c_pa)
    group['BB%'] = safe_div(c_bb, c_pa)
    group['K/BB'] = safe_div(c_k, c_bb)
    group['Contact%'] = safe_div((c_pa - c_k), c_pa)
    group['sweet_spot'] = (group['launch_angle'].between(8, 32).fillna(False) & group['is_bip'].astype(bool)).astype(int)
    c_sweet = group['sweet_spot'].cumsum().shift(1).fillna(0)
    group['SweetSpot%'] = safe_div(c_sweet, c_bip)
    group['hard_hit'] = ((group['launch_speed'] >= 95).fillna(False) & group['is_bip'].astype(bool)).astype(int)
    c_hard = group['hard_hit'].cumsum().shift(1).fillna(0)
    group['HardHit%'] = safe_div(c_hard, c_bip)
    group['xwoba_est'] = (
        0.7*group['is_bb'] +
        0.9*(group['events'] == 'single').astype(int) +
        1.3*(group['events'] == 'double').astype(int) +
        1.6*(group['events'] == 'triple').astype(int) +
        2.0*(group['events'] == 'home_run').astype(int)
    )
    c_xwoba = group['xwoba_est'].cumsum().shift(1).fillna(0)
    group['xwOBA'] = safe_div(c_xwoba, c_pa)
    
    group['team'] = group['away_team'].where(group['inning_topbot'] == 'Top', group['home_team'])

    # Fetch lineup position
    group['lineup_pos'] = np.nan
    # Iterate through all games for that player
    for gdate in group['game_date'].unique():
        mask = group['game_date'] == gdate
        game_date_str = pd.to_datetime(gdate).strftime('%Y-%m-%d')
        batter_id = group.loc[mask, 'batter'].iloc[0]
        side = 'away_lineup' if group.loc[mask, 'inning_topbot'].iloc[0] == 'Top' else 'home_lineup'
        lineup_pos = get_lineup_pos(batter_id, lineups[lineups['date'] == game_date_str], side)
        group.loc[mask, 'lineup_pos'] = lineup_pos
    
    # Final features
    group = group[['game_date', 'batter', 'team', 'AVG', 'SLG', 'OBP', 'HR', 'xwOBA', 'K%', 'BB%', 'K/BB', 'Contact%', 'SweetSpot%', 'HardHit%', 'lineup_pos']]
    # Remove any rows where lineup_pos is not found - pinch hitters
    group = group[group['lineup_pos'].notna()]
    # Only keep one row per player per game
    group = group.groupby(['game_date', 'batter']).first().reset_index()
    # Drop first 10 games of season to allow for normalization of features
    group = group.iloc[10:]
    features_df.append(group)

features_df = pd.concat(features_df)
features_df.rename(columns={'batter': 'player_id'}, inplace=True)
features_df = features_df.sort_values(['game_date', 'team'])
print("Features saved to features_df")

Calculating features: 100%|██████████| 460/460 [00:32<00:00, 14.09it/s]

Features saved to features_df


## Dataset Construction

After feature engineering, we construct a dataset where each row represents a **player-game instance**, including:
- Player ID
- Team
- Game date
- Performance features
- Observed lineup position (target variable)

To ensure realistic evaluation, we split the data using a grouped train-test split, where all players from a single game are kept together. This prevents leakage between training and validation sets and ensures that full lineups are only seen in one split.

We also normalize features using standard scaling to improve model training stability.


## Player-Level Model

We first train a neural network to predict lineup position for individual players.

This model takes in a vector of player performance features and outputs a probability distribution over the nine lineup positions using a softmax layer.

The architecture consists of:
- Two fully connected layers
- A learned embedding layer (32-dimensional representation)
- A final output layer with 9 logits

We train the model using categorical cross-entropy loss and the Adam optimizer, with L2 regularization to prevent overfitting. This model allows us to capture uncertainty in lineup placement, as players may occupy different positions throughout the season.

We evaluate the player model using several metrics:

- Top-1 Accuracy: Exact prediction of lineup position
- Top-3 Accuracy: Whether the true position is among the top 3 predicted probabilities
- Mean Absolute Error (MAE): Average positional distance between prediction and truth

Top-3 accuracy is particularly important, as lineup positions are ordinal and small errors (e.g., predicting 2 instead of 1) are less severe than larger ones. We also analyze prediction outputs and save them for further use in downstream modeling.

## Team-Level Model

While the player model predicts positions independently, lineup construction is inherently a joint problem involving all nine players. To address this, we build a team-level model that takes all nine players in a lineup as input and predicts their positions simultaneously.

This model leverages the embeddings learned by the player model, using them as input features. By doing so, it captures richer representations of player skill while modeling interactions between players.

The model outputs a matrix of scores representing each player’s suitability for each lineup position.

To convert model outputs into a valid lineup (each position assigned exactly once), we apply the Hungarian algorithm (linear sum assignment). This ensures:
- Each player is assigned exactly one position
- Each lineup position is filled exactly once
- The overall assignment maximizes model confidence

We evaluate the team model using several metrics:
- Exact Match Accuracy: Percentage of perfectly predicted lineups
- Average Displacement: Average positional error across players
- Pairwise Accuracy: How often the model correctly predicts which player bats before another

We analyze position-specific accuracy and construct a confusion matrix to understand common prediction errors, and also use permutation importance to understand which features drive predictions.

## Results, Insights, and Conclusions

Our results show that lineup position can be predicted with meaningful accuracy using player performance metrics.

Key observations include:
- Certain positions (e.g., beginning of the lineup) are easier to predict than others
- The model captures general lineup trends
- Joint modeling improves coherence of predicted lineups compared to independent predictions

However, performance is limited by factors not captured in the data, such as managerial preferences, injuries, and game-specific strategy. This work highlights both the potential and limitations of data-driven approaches to baseball strategy, and provides a foundation for future improvements incorporating additional context and data.

